# Average Bootstrap Test Analysis

This notebook performs statistical testing of CRE (Cis-Regulatory Element) activity across cell types using bootstrap resampling.

## Analysis Overview:
1. **Bootstrap Testing**: Estimate cell type-specific CRE activity with uncertainty quantification
2. **Reproducibility Analysis**: Compare results between tissue sections (Section 1 vs Section 2)
3. **Technical Validation**: Assess correlation with ATAC-seq and histone modification data
4. **Quality Control**: Identify and filter low-confidence CRE-cell type associations

## Key Outputs:
- Q-values for each CRE-cell type pair
- Reproducibility metrics across biological replicates
- ATAC-seq precision and recall statistics
- Spatial visualizations of significant CREs

## 1. Setup and Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="docrep")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import sys
import os
import re
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from upsetplot import UpSet, from_contents

scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [2]:
# Add current path to sys.path
try:
    PWD = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PWD = '/gpfs/commons/groups/ren_lab/guojiezhong/starr-fish/Mouse_brain.Guojie'
sys.path.append(f'{PWD}/')
os.chdir(PWD)

from STARRFISH import STARRFISH
from STARRFISH.plots import plot_grouped_clustermap, celltype_pval_dotplot

Seed set to 0


Last run with scvi-tools version: 1.3.0


## 2. Helper Functions

In [3]:
def reload(starrfish):
    """Reload STARRFISH modules and update class references."""
    import importlib
    import STARRFISH
    importlib.reload(STARRFISH)
    from STARRFISH import STARRFISH
    starrfish.__class__ = STARRFISH
    return starrfish

def drop_test(starrfish, test_method):
    """Remove test results from starrfish object."""
    if hasattr(starrfish, f'{test_method}_configs'):
        delattr(starrfish, f'{test_method}_configs')
    if hasattr(starrfish, f'{test_method}_results'):
        delattr(starrfish, f'{test_method}_results')
    return starrfish

def preprocess(adata_path):
    """Preprocess AnnData object with CRE and cell type annotations."""
    if type(adata_path) is str:
        adata = sc.read_h5ad(adata_path)
    elif type(adata_path) is sc.AnnData:
        adata = adata_path
    
    # Extract FOV from index
    adata.obs['fov'] = adata.obs.index.str.split('--').str[0]
    
    # Clean cell type names
    adata.obs['subclass'] = adata.obs['subclass_name'].str.replace('^[0-9]+ ', '', regex=True)
    adata.obs['class'] = adata.obs['class_name'].str.replace('^[0-9]+ ', '', regex=True)
    
    # Fill empty best_subclass
    adata.uns['CRE_info']['best_subclass'][adata.uns['CRE_info']['best_subclass'] == ''] = \
        adata.uns['CRE_info']['label'][adata.uns['CRE_info']['best_subclass'] == ''].copy()
    
    # Process genomic coordinates
    chrom, start, end = [], [], []
    for i in adata.uns['CRE_info']['enh']:
        if i.startswith('chr'):
            chrom.append(i.split(':')[0])
            start.append(int(re.split('−|-', i.split(':')[1])[0]))
            end.append(int(re.split('−|-', i.split(':')[1])[1]))
        else:
            chrom.append(i)
            start.append('')
            end.append('')
    
    adata.uns['CRE_info']['Chrom'] = pd.Series(chrom).astype(str)
    adata.uns['CRE_info']['Start'] = pd.Series(start).astype(str)
    adata.uns['CRE_info']['End'] = pd.Series(end).astype(str)
    adata.uns['CRE_info']['enh'] = adata.uns['CRE_info']['Chrom'] + ':' + \
        adata.uns['CRE_info']['Start'] + '-' + adata.uns['CRE_info']['End']
    
    # Clean names
    adata.uns['CRE_info']['best_subclass'] = adata.uns['CRE_info']['best_subclass'].str.replace('_', ' ')
    adata.uns['CRE_info'].index = ['CRE' + str(i+1).zfill(3) for i in range(len(adata.uns['CRE_info']))]
    adata.obsm['CRE'] = adata.obsm['CRE'][adata.uns['CRE_info'].index]
    
    if 'T7CRE' in adata.obsm.keys():
        adata.obsm['T7CRE'] = adata.obsm['T7CRE'][adata.uns['CRE_info'].index]
    
    return adata

## 3. Data Loading and Configuration

Load STARRFISH objects for two tissue sections and the combined dataset. Apply quality control filters.

In [4]:
# Load section 1 data
starrfish3_sec1 = STARRFISH.load('results/starrfish3_sec1.pkl')

# Load cell type annotations
subclass_annotation = pd.read_excel(f'Data/abc_atlas/allen_institute_nominature.xlsx')
subclass_annotation['subclass'] = subclass_annotation['subclass_id_label'].str.replace('^[0-9]+ ', '', regex=True)
subclass_annotation['subclass'] = subclass_annotation['subclass'].str.replace('/', '-', regex=True)
subclass_to_subclass_name = subclass_annotation['subclass_id_label'].groupby(
    subclass_annotation['subclass']).first().to_dict()
subclass_name_to_subclass = subclass_annotation['subclass'].groupby(
    subclass_annotation['subclass_id_label']).first().to_dict()

# Define CRE blacklist
cre_blacklist = ['CRE061', 'CRE143', 'CRE001']
cre_whitelist = starrfish3_sec1.get_creinfo().index[~starrfish3_sec1.get_creinfo().index.isin(cre_blacklist)]

# Filter by barcode mismatch rate
mismatching_cres = pd.read_csv('Data/AAV_ONT_Barcode_Counts_vs_Mismatch_Percentage.csv', index_col=0)
cre_whitelist = cre_whitelist[~cre_whitelist.isin(mismatching_cres.index[mismatching_cres['MismatchPercent'] > 20])]
cre_blacklist = np.unique(cre_blacklist + mismatching_cres.index[mismatching_cres['MismatchPercent'] > 20].tolist()).tolist()

print(f"Total CREs in blacklist: {len(cre_blacklist)}")
print(f"Total CREs in whitelist: {len(cre_whitelist)}")

Total CREs in blacklist: 11
Total CREs in whitelist: 389


## 4. Bootstrap Test Configuration

Configure bootstrap resampling parameters for statistical testing.

In [5]:
average_bootstrap_test_config = {
    'cell_types_to_use': None,
    'normalize_by_cell_rna': False,
    'normalize_by_cell_volume': False,
    'normalize_by_cell_t7': False,
    'normalize_by_celltype_rna': False,
    'normalize_by_celltype_volume': False,
    'normalize_by_celltype_t7': True,  # Normalize by T7 at cell type level
    'filter_by_cell_t7': None,
    'normalize_by_negative_control': False,
    'normalize_by_libsize': False,
    'log_transform': False,
    'bootstrap_number': 10000,  # Number of bootstrap iterations
    'bootstrap_to_fixed_pct': 1,
    'bootstrap_to_fixed_sample_size': None,
    'load_stored': True,  # Load previously computed results if available
    'n_jobs': 62,  # Parallel processing cores
}

infected_cells_threshold = 5  # Minimum cells with CRE detection
threshold = 'neg_control_mean'  # Statistical threshold method

## 5. Run Bootstrap Tests

Execute bootstrap testing for Section 1, Section 2, and combined dataset.

In [6]:
# Section 1
# load pre-computed results
starrfish3_sec1 = STARRFISH.load('results/starrfish3_sec1.bak.pkl')
res1 = starrfish3_sec1.average_bootstrap_test(**average_bootstrap_test_config)
del starrfish3_sec1

starrfish3_sec1 = STARRFISH.load('results/starrfish3_sec1.pkl')
to_filter_sec1 = (starrfish3_sec1.get_cre_expression() > 0).groupby(
    starrfish3_sec1.get_celltypes()).sum() < infected_cells_threshold
to_filter_sec1[cre_blacklist] = True
res_q1, res_df1, _ = starrfish3_sec1.average_bootstrap_test_q(
    res1, threshold=threshold, norm='T7', tail='both', to_filter=to_filter_sec1, calibrate=None)
# add blacklist cres
starrfish3_sec1.blacklist_cre = cre_blacklist
print(f"Section 1: {res_df1.shape[0]} cell types, {res_df1.shape[1]} CREs")

Results already exist, return stored results
Section 1: 303 cell types, 400 CREs


In [7]:
# Section 2
# load pre-computed results
starrfish3_sec2 = STARRFISH.load('results/starrfish3_sec2.bak.pkl')
res2 = starrfish3_sec2.average_bootstrap_test(**average_bootstrap_test_config)
del starrfish3_sec2

starrfish3_sec2 = STARRFISH.load('results/starrfish3_sec2.pkl')
to_filter_sec2 = (starrfish3_sec2.get_cre_expression() > 0).groupby(
    starrfish3_sec2.get_celltypes()).sum() < infected_cells_threshold
to_filter_sec2[cre_blacklist] = True
res_q2, res_df2, _ = starrfish3_sec2.average_bootstrap_test_q(
    res2, threshold=threshold, norm='T7', tail='both', to_filter=to_filter_sec2, calibrate=None)
# add blacklist cres
starrfish3_sec2.blacklist_cre = cre_blacklist
print(f"Section 2: {res_df2.shape[0]} cell types, {res_df2.shape[1]} CREs")

Results already exist, return stored results
Section 2: 319 cell types, 400 CREs


In [8]:
# Combined dataset
# load pre-computed results
starrfish3 = STARRFISH.load('results/starrfish3.bak.pkl')
res = starrfish3.average_bootstrap_test(**average_bootstrap_test_config)
del starrfish3

starrfish3 = STARRFISH.load('results/starrfish3.pkl')
to_filter = (starrfish3.get_cre_expression() > 0).groupby(
    starrfish3.get_celltypes()).sum() < infected_cells_threshold
to_filter[cre_blacklist] = True
res_q, res_df, _ = starrfish3.average_bootstrap_test_q(
    res, threshold=threshold, norm='T7', tail='both', to_filter=to_filter, calibrate=None)
# add blacklist cres
starrfish3.blacklist_cre = cre_blacklist
print(f"Combined: {res_df.shape[0]} cell types, {res_df.shape[1]} CREs")

Results already exist, return stored results
Combined: 328 cell types, 400 CREs


## 6. Technical Bias Analysis

### 6.1 T7 Total Count Correlation

In [9]:
# Check for bias related to total T7 counts per CRE
toplot = pd.DataFrame({
    'value': res_df.values.flatten(), 
    'T7 total': np.tile(starrfish3.get_t7_expression().sum().values, res_df.shape[0])
})

fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(data=toplot, x='T7 total', y='value', alpha=0.1, ax=ax)

# Overlay average values
toplot_avg = res_df.apply(np.nanmean, axis=0)
sns.scatterplot(x=starrfish3.get_t7_expression().sum().values, y=toplot_avg.values, 
                color='red', ax=ax, alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('Total T7 counts (per CRE)')
ax.set_ylabel('Activity per CRE/Cell Type pair')
ax.set_title('CRE Activity vs T7 Detection')
plt.show()

### 6.2 Cell Type T7 Count Bias

In [10]:
# Check for cell type-specific bias
toplot = pd.DataFrame({
    'value': res_df.values.flatten(), 
    'T7 total': np.repeat(starrfish3.get_t7_expression().sum(axis=1).groupby(
        starrfish3.get_celltypes()).sum().values, res_df.shape[1])
})

fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(data=toplot, x='T7 total', y='value', alpha=0.1, ax=ax)

toplot_avg = res_df.apply(np.nanmean, axis=1)
sns.scatterplot(x=starrfish3.get_t7_expression().sum(axis=1).groupby(
    starrfish3.get_celltypes()).sum().values, y=toplot_avg.values, 
    color='red', ax=ax, alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('Total T7 counts (per Cell Type)')
ax.set_ylabel('Activity per CRE/Cell Type pair')
ax.set_title('Cell Type Activity vs T7 Detection')
plt.show()

## 7. Visualization: Heatmap

Generate clustered heatmap of CRE activity across cell types.

In [11]:
# Prepare CRE info for annotation
cre_info = starrfish3.get_creinfo().copy()
cre_info['best_subclass'] = 'CRE'
cre_info.loc[starrfish3.get_negative_control_cres(), 'best_subclass'] = 'Negative Control'

# Plot heatmap (only showing CREs/cell types with some NA values)
_, final_order = plot_grouped_clustermap(
    res_df.loc[pd.isna(res_df).any(axis=1), pd.isna(res_df).any(axis=0)], 
    cre_info, 'All', figsize=(15, 8)
)
plt.show()

## 8. Calculate Q-values with Calibration

Recompute q-values with self-CRE calibration for all tests (including left and right tails).

In [12]:
# Combined dataset with calibration
to_filter = (starrfish3.get_cre_expression() > 0).groupby(
    starrfish3.get_celltypes()).sum() < 0  # No filtering for visualization
to_filter[cre_blacklist] = True

res_q, res_q_right, res_q_left, res_df, res_df_fdc = starrfish3.average_bootstrap_test_q(
    res, threshold=threshold, norm='T7', tail='all', 
    to_filter=to_filter, calibrate='self-CRE'
)

print(f"Q-values calculated for {res_q.shape[0]} cell types and {res_q.shape[1]} CREs")

Q-values calculated for 328 cell types and 400 CREs


In [13]:
# Section 1
to_filter_sec1 = (starrfish3_sec1.get_cre_expression() > 0).groupby(
    starrfish3_sec1.get_celltypes()).sum() < 0
to_filter_sec1[cre_blacklist] = True

res_q1, res_q1_right, res_q1_left, res_df1, res_df1_fdc = starrfish3.average_bootstrap_test_q(
    res1, threshold=threshold, norm='T7', tail='all',
    to_filter=to_filter_sec1, calibrate='self-CRE'
)

In [14]:
# Section 2
to_filter_sec2 = (starrfish3_sec2.get_cre_expression() > 0).groupby(
    starrfish3_sec2.get_celltypes()).sum() < 0
to_filter_sec2[cre_blacklist] = True

res_q2, res_q2_right, res_q2_left, res_df2, res_df2_fdc = starrfish3.average_bootstrap_test_q(
    res2, threshold=threshold, norm='T7', tail='all',
    to_filter=to_filter_sec2, calibrate='self-CRE'
)

In [15]:
# write intrinsic activity and significance to file
res_df.to_csv('results/intrinsic_activity.csv')
res_q_right.to_csv('results/intrinsic_activity_qvalues.csv')

## 9. Reproducibility Analysis

### 9.1 Correlation Between Sections

In [16]:
# Calculate Pearson correlation between sections
cre_corr, celltype_corr = starrfish3.corr_starrfish(res_df1, res_df2)

# Add library size info
cre_corr['libsize'] = starrfish3.lib_size['counts'].loc[cre_corr.index]

# Add cell counts
celltype_corr['celltype_sec1'] = starrfish3_sec1.get_celltypes().value_counts().reindex(
    celltype_corr.index).fillna(0).values
celltype_corr['celltype_sec2'] = starrfish3_sec2.get_celltypes().value_counts().reindex(
    celltype_corr.index).fillna(0).values
celltype_corr['celltype_full'] = starrfish3.get_celltypes().value_counts().reindex(
    celltype_corr.index).fillna(0).values
celltype_corr['celltype_n'] = np.minimum(
    celltype_corr['celltype_sec1'], celltype_corr['celltype_sec2'])

print(f"Median CRE correlation: {cre_corr['pearson'].median():.3f}")
print(f"Median cell type correlation: {celltype_corr['pearson'].median():.3f}")

Error in calculating correlation for CRE:  CRE001
Error in calculating correlation for CRE:  CRE010
Error in calculating correlation for CRE:  CRE011
Error in calculating correlation for CRE:  CRE021
Error in calculating correlation for CRE:  CRE032
Error in calculating correlation for CRE:  CRE033
Error in calculating correlation for CRE:  CRE060
Error in calculating correlation for CRE:  CRE061
Error in calculating correlation for CRE:  CRE062
Error in calculating correlation for CRE:  CRE071
Error in calculating correlation for CRE:  CRE072
Error in calculating correlation for CRE:  CRE074
Error in calculating correlation for CRE:  CRE075
Error in calculating correlation for CRE:  CRE087
Error in calculating correlation for CRE:  CRE089
Error in calculating correlation for CRE:  CRE093
Error in calculating correlation for CRE:  CRE102
Error in calculating correlation for CRE:  CRE103
Error in calculating correlation for CRE:  CRE104
Error in calculating correlation for CRE:  CRE111


### 9.2 Cell Type Correlation Plots (Figure 4b)

In [17]:
n_cre_threshold = 20
n_celltype_threshold = 5

fig, ax = plt.subplots(figsize=(4, 4))
# Non-significant correlations in blue
sns.scatterplot(data=celltype_corr[(celltype_corr['pearson_p'] > 0.05) & 
                                    (celltype_corr['effect_n'] >= n_cre_threshold)], 
                x='celltype_n', y='pearson', color='blue', ax=ax)
# Significant correlations in red
sns.scatterplot(data=celltype_corr[(celltype_corr['pearson_p'] <= 0.05) & 
                                    (celltype_corr['effect_n'] >= n_cre_threshold)], 
                x='celltype_n', y='pearson', color='red', ax=ax)
ax.set_xscale('log')
ax.set_xlabel('Number of cells (minimum between sections)')
ax.set_ylabel('Pearson correlation')
ax.set_title('Cell Type Reproducibility')
fig.savefig('results/expr3/reproducibility_by_celltype_pearson_sec1_sec2.pdf')
plt.show()

In [18]:
# Violin plot for well-sampled cell types
fig, ax = plt.subplots(figsize=(2, 4))
sns.violinplot(data=celltype_corr[(celltype_corr['effect_n'] >= n_cre_threshold) & 
                                   (celltype_corr['celltype_n'] >= 1000)], 
               y='pearson', ax=ax)
ax.set_ylabel('Pearson correlation')
ax.set_title('Reproducible Cell Types\n(n≥1000 cells)')

reproducible_celltypes = celltype_corr.index[
    (celltype_corr['effect_n'] >= n_cre_threshold) & 
    (celltype_corr['celltype_n'] >= 1000)
]
fig.savefig('results/expr3/reproducibility_by_celltype_pearson_violin_sec1_sec2.pdf')
plt.show()

print(f"Number of reproducible cell types: {len(reproducible_celltypes)}")

Number of reproducible cell types: 26


### 9.3 CRE Correlation Plots (Figure 4c)

In [19]:
n_cre_threshold = 20
n_celltype_threshold = 20

fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(data=cre_corr[(cre_corr['pearson_p'] > 0.05) & 
                              (cre_corr['effect_n'] >= n_celltype_threshold)], 
                x='libsize', y='pearson', color='blue', ax=ax)
sns.scatterplot(data=cre_corr[(cre_corr['pearson_p'] <= 0.05) & 
                              (cre_corr['effect_n'] >= n_celltype_threshold)], 
                x='libsize', y='pearson', color='red', ax=ax)
ax.set_xscale('log')
ax.set_xlabel('AAV library size')
ax.set_ylabel('Pearson correlation')
ax.set_title('CRE Reproducibility vs Library Size')
fig.savefig('results/expr3/reproducibility_by_cre_pearson_sec1_sec2.pdf')
plt.show()

In [20]:
fig, ax = plt.subplots(figsize=(2, 4))
sns.violinplot(data=cre_corr[(cre_corr['effect_n'] >= n_celltype_threshold)], 
               y='pearson', ax=ax)
ax.set_ylabel('Pearson correlation')
ax.set_title('CRE Reproducibility Distribution')
fig.savefig('results/expr3/reproducibility_by_cre_pearson_violin_sec1_sec2.pdf')
plt.show()

## 10. Overlap Statistics

Calculate overlap of significant CREs between sections.

In [21]:
# Get overlapping data (only cell types present in all datasets)
res_q2_overlap = res_q2_right[(~res_q2_right.isna()) & (~res_q1_right.isna()) & (~res_q_right.isna())].copy()
res_q1_overlap = res_q1_right[(~res_q2_right.isna()) & (~res_q1_right.isna()) & (~res_q_right.isna())].copy()
res_q_overlap = res_q_right[(~res_q2_right.isna()) & (~res_q1_right.isna()) & (~res_q_right.isna())].copy()

overlap_df = pd.DataFrame(
    index=res_q2_overlap.index.intersection(res_q1_overlap.index),
    columns=['sec1', 'sec2', 'all', 'overlap_sec1_sec2', 'overlap_sec1_all', 
             'overlap_sec2_all', 'percentage_sec1_sec2', 'percentage_sec1_all', 'percentage_sec2_all']
)

# Count significant CREs per cell type
overlap_df['sec1'] = (res_q1_overlap.loc[overlap_df.index] <= 0.05).sum(axis=1)
overlap_df['sec2'] = (res_q2_overlap.loc[overlap_df.index] <= 0.05).sum(axis=1)
overlap_df['all'] = (res_q_overlap.loc[overlap_df.index] <= 0.05).sum(axis=1)

# Count overlapping significant CREs
overlap_df['overlap_sec1_sec2'] = ((res_q1_overlap.loc[overlap_df.index] <= 0.05) & 
                                   (res_q2_overlap.loc[overlap_df.index] <= 0.05)).sum(axis=1)
overlap_df['overlap_sec1_all'] = ((res_q1_overlap.loc[overlap_df.index] <= 0.05) & 
                                  (res_q_overlap.loc[overlap_df.index] <= 0.05)).sum(axis=1)
overlap_df['overlap_sec2_all'] = ((res_q2_overlap.loc[overlap_df.index] <= 0.05) & 
                                  (res_q_overlap.loc[overlap_df.index] <= 0.05)).sum(axis=1)

# Calculate percentages
overlap_df['percentage_sec1_sec2'] = overlap_df['overlap_sec1_sec2'] / np.minimum(
    overlap_df['sec1'], overlap_df['sec2'])
overlap_df['percentage_sec1_all'] = overlap_df['overlap_sec1_all'] / np.minimum(
    overlap_df['sec1'], overlap_df['all'])
overlap_df['percentage_sec2_all'] = overlap_df['overlap_sec2_all'] / np.minimum(
    overlap_df['sec2'], overlap_df['all'])

overlap_df = overlap_df.sort_values('percentage_sec1_sec2', ascending=False)

# Add cell counts
overlap_df['celltype_n_sec1'] = starrfish3_sec1.get_celltypes().value_counts().reindex(
    overlap_df.index).fillna(0).astype(int).values
overlap_df['celltype_n_sec2'] = starrfish3_sec2.get_celltypes().value_counts().reindex(
    overlap_df.index).fillna(0).astype(int).values
overlap_df['celltype_n_all'] = starrfish3.get_celltypes().value_counts().reindex(
    overlap_df.index).fillna(0).astype(int).values
overlap_df['celltype_n'] = np.minimum(
    overlap_df['celltype_n_sec1'], overlap_df['celltype_n_sec2'], overlap_df['celltype_n_all'])

print(f"Mean reproducibility (Sec1-Sec2): {overlap_df['percentage_sec1_sec2'].mean():.3f}")

Mean reproducibility (Sec1-Sec2): 0.293


/gpfs/commons/home/guojiezhong/tmp/ipykernel_4175689/3890892211.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  overlap_df['celltype_n'] = np.minimum(


### 10.1 Reproducibility Visualization Plots (Figure S8d)

In [22]:
def plot_reproducibility(overlap_df, celltypes_to_use, percentage_col, bar1_col, bar2_col, 
                        bar1_label, bar2_label):
    """Plot reproducibility metrics across cell types.
    
    Parameters:
    -----------
    overlap_df : DataFrame
        Overlap statistics for each cell type
    celltypes_to_use : Index
        Cell types to include in plot
    percentage_col : str
        Column name for reproducibility percentage
    bar1_col, bar2_col : str
        Columns for bar plot comparison
    bar1_label, bar2_label : str
        Labels for bar plots
    """
    # Order by Allen Institute nomenclature
    cluster_annotation_term = pd.read_csv('Data/abc_atlas/cluster_annotation_term.csv', index_col=0)
    cluster_annotation_term['subclass'] = cluster_annotation_term['subclass'].str.replace('/', '-')
    overlap_df['cell_type_rank'] = cluster_annotation_term['subclass_number'].groupby(
        cluster_annotation_term['subclass']).first().loc[overlap_df.index].values
    overlap_df = overlap_df.sort_values(['cell_type_rank'], ascending=[True])
    
    # Filter for well-sampled cell types
    overlap_df_filtered = overlap_df[overlap_df['celltype_n'] >= 1000]
    overlap_df_filtered = overlap_df_filtered.loc[celltypes_to_use.intersection(overlap_df_filtered.index)]
    overlap_df_filtered = overlap_df_filtered.sort_values(['cell_type_rank'], ascending=[True])
    
    cell_types = overlap_df_filtered.index
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Left y-axis: reproducibility percentage
    sns.barplot(x=cell_types, y=overlap_df_filtered[percentage_col], ax=ax1, alpha=0.8)
    ax1.set_xlabel('Cell Types')
    ax1.set_ylabel('Reproducibility (percentage)', color='black')
    ax1.tick_params(axis='y', labelcolor='black')
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    
    # Right y-axis: CRE counts
    ax2 = ax1.twinx()
    x_pos = np.arange(len(overlap_df_filtered))
    
    color1 = 'orange' if bar1_col == 'sec1' else 'green' if bar1_col == 'sec2' else 'pink'
    color2 = 'orange' if bar2_col == 'sec1' else 'green' if bar2_col == 'sec2' else 'pink'
    
    ax2.bar(x_pos - 0.2, overlap_df_filtered[bar1_col], 0.4, 
            label=bar1_label, alpha=0.8, color=color1)
    ax2.bar(x_pos + 0.2, overlap_df_filtered[bar2_col], 0.4, 
            label=bar2_label, alpha=0.8, color=color2)
    ax2.set_ylabel('# significant CREs', color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    return fig

In [23]:
# Plot Sec1-Sec2 reproducibility
fig = plot_reproducibility(overlap_df, reproducible_celltypes, 'percentage_sec1_sec2', 
                           'sec1', 'sec2', 'sec1', 'sec2')
fig.savefig('results/expr3/reproducibility_by_celltype_sec1_sec2.pdf')
plt.show()

# Plot Sec1-All reproducibility
fig = plot_reproducibility(overlap_df, reproducible_celltypes, 'percentage_sec1_all', 
                           'sec1', 'all', 'sec1', 'all')
fig.savefig('results/expr3/reproducibility_by_celltype_sec1_all.pdf')
plt.show()

# Plot Sec2-All reproducibility
fig = plot_reproducibility(overlap_df, reproducible_celltypes, 'percentage_sec2_all', 
                           'sec2', 'all', 'sec2', 'all')
fig.savefig('results/expr3/reproducibility_by_celltype_sec2_all.pdf')
plt.show()

### 10.2 CRE-Level Overlap Plot (Figure S8e)

In [24]:
def calculate_overlap_cre_df(q_values_df1, q_values_df2, comparison_name1='group1', 
                             comparison_name2='group2', cre_blacklist=None):
    """Calculate overlap statistics between two q-value dataframes.
    
    Returns DataFrame with overlap statistics and reproducibility categories.
    """
    if cre_blacklist is None:
        cre_blacklist = []
    
    overlap_cre_df = pd.DataFrame(
        index=q_values_df1.columns,
        columns=['no_na', comparison_name1, comparison_name2, 'overlap', 'percentage']
    )
    
    for cre in q_values_df1.columns:
        if comparison_name2 == comparison_name1:
            overlap_cre_df.loc[cre, 'no_na'] = sum(~pd.isna(q_values_df1[cre]))
        else:
            overlap_cre_df.loc[cre, 'no_na'] = sum(
                ~pd.isna(q_values_df1[cre]) & ~pd.isna(q_values_df2[cre]))
        
        overlap_cre_df.loc[cre, comparison_name1] = sum(q_values_df1[cre] <= 0.05)
        overlap_cre_df.loc[cre, comparison_name2] = sum(q_values_df2[cre] <= 0.05)
        overlap_cre_df.loc[cre, 'overlap'] = sum(
            (q_values_df1[cre] <= 0.05) & (q_values_df2[cre] <= 0.05))
        
        if overlap_cre_df.loc[cre, comparison_name1] == 0 and overlap_cre_df.loc[cre, comparison_name2] == 0:
            if overlap_cre_df.loc[cre, 'no_na'] > 0:
                overlap_cre_df.loc[cre, 'percentage'] = 1
            else:
                overlap_cre_df.loc[cre, 'percentage'] = -1
        else:
            overlap_cre_df.loc[cre, 'percentage'] = overlap_cre_df.loc[cre, 'overlap'] / np.maximum(
                overlap_cre_df.loc[cre, comparison_name1], overlap_cre_df.loc[cre, comparison_name2])
    
    overlap_cre_df['reproducibility'] = 'Non-reproducible'
    overlap_cre_df.loc[overlap_cre_df['percentage'] == 1, 'reproducibility'] = 'All Reproducible'
    overlap_cre_df.loc[overlap_cre_df['percentage'] == -1, 'reproducibility'] = 'All NA'
    overlap_cre_df.loc[(overlap_cre_df['percentage'] > 0) & 
                       (overlap_cre_df['percentage'] < 1), 'reproducibility'] = 'Partially Reproducible'
    overlap_cre_df.loc[cre_blacklist, 'reproducibility'] = 'Blacklisted'
    
    return overlap_cre_df

In [25]:
# Calculate CRE-level overlap
res_q1_overlap = res_q1_overlap.loc[overlap_df.index]
res_q2_overlap = res_q2_overlap.loc[overlap_df.index]
res_q_overlap = res_q_overlap.loc[overlap_df.index]

overlap_cre_df = calculate_overlap_cre_df(res_q1_overlap, res_q2_overlap, 'sec1', 'sec2', cre_blacklist)
overlap_cre_df_sec1_all = calculate_overlap_cre_df(res_q1_overlap, res_q_overlap, 'sec1', 'all', cre_blacklist)
overlap_cre_df_sec2_all = calculate_overlap_cre_df(res_q2_overlap, res_q_overlap, 'sec2', 'all', cre_blacklist)

print("Sec1-Sec2 overlap statistics:")
print(f"  Reproducible: {sum(overlap_cre_df['percentage'] > 0)}")
print(f"  All NA: {sum(overlap_cre_df['percentage'] == -1)}")
print(f"  Perfect reproducibility: {sum(overlap_cre_df['percentage'] == 1)}")

Sec1-Sec2 overlap statistics:
  Reproducible: 127
  All NA: 46
  Perfect reproducibility: 53


In [26]:
# Plot reproducibility categories
toplot = pd.concat((
    overlap_cre_df['reproducibility'].value_counts().rename('sec1_sec2'),
    overlap_cre_df_sec1_all['reproducibility'].value_counts().rename('sec1_all'),
    overlap_cre_df_sec2_all['reproducibility'].value_counts().rename('sec2_all')
), axis=1)

# Merge 'All NA' into 'All Reproducible'
toplot.loc['All Reproducible'] = toplot.loc['All Reproducible'].fillna(0) + toplot.loc['All NA'].fillna(0)
toplot = toplot.drop(index='All NA', errors='ignore')

# Reshape for plotting
toplot_reset = toplot.reset_index()
index_col_name = toplot_reset.columns[0]
toplot_melted = toplot_reset.melt(id_vars=index_col_name, var_name='comparison', value_name='count')
toplot_melted = toplot_melted.rename(columns={index_col_name: 'reproducibility'})

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=toplot_melted[toplot_melted['reproducibility'] != 'Non-reproducible'], 
            x='comparison', y='count', hue='reproducibility', ax=ax)
ax.set_title('CRE Reproducibility Across Comparisons')
ax.set_xlabel('Comparison')
ax.set_ylabel('Count')
ax.legend(title='Reproducibility Category', bbox_to_anchor=(1.05, 1), loc='upper left')
fig.tight_layout()
fig.savefig('results/expr3/reproducibility_comparison_barplot.pdf')
plt.show()

## 11. Dot Plots: CRE Activity Dot plots (Figure 4d, S8c)

Generate comprehensive dot plots showing significant CRE-cell type associations.

In [27]:
# Complete dataset
cell_types_to_use = res_q1_right.index.intersection(res_q2_right.index)
cres_to_use = res_q_right.columns[
    np.nanmin(res_q_right.loc[cell_types_to_use], axis=0) < 0.05
].union(starrfish3.get_negative_control_cres())
cres_to_use = cres_to_use[~cres_to_use.isin(cre_blacklist)]

fig, final_order = celltype_pval_dotplot(
    res_q_right, res_df_fdc, cres_to_use, cell_types_to_use,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(50, 30)
)
fig.savefig('results/expr3/celltype_pval_dotplot_complete.pdf')
plt.close()

In [28]:
# Well-sampled cell types only
cell_types_to_use = starrfish3.get_celltypes().value_counts().index[
    starrfish3.get_celltypes().value_counts() >= 1000
]
cres_to_use = res_q_right.columns[
    np.nanmin(res_q_right.loc[cell_types_to_use], axis=0) < 0.05
].union(starrfish3.get_negative_control_cres())
cres_to_use = cres_to_use[~cres_to_use.isin(cre_blacklist)]

fig, final_order = celltype_pval_dotplot(
    res_q_right, res_df_fdc, cres_to_use, cell_types_to_use,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(20, 30)
)
fig.savefig('results/expr3/celltype_pval_dotplot_all.pdf')
plt.close()

In [29]:
# Section 1 (same CRE order)
fig, _ = celltype_pval_dotplot(
    res_q1_right, res_df1_fdc, pd.Index(final_order), cell_types_to_use, reorder_cres=False,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(20, 30)
)
fig.savefig('results/expr3/celltype_pval_dotplot_sec1.pdf')
plt.close()

# Section 2 (same CRE order)
fig, _ = celltype_pval_dotplot(
    res_q2_right, res_df2_fdc, pd.Index(final_order), cell_types_to_use, reorder_cres=False,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(20, 30)
)
fig.savefig('results/expr3/celltype_pval_dotplot_sec2.pdf')
plt.close()

## 12. Spatial Plots of Top CREs (Figure 4e, 4f, 4g, 4h)

Generate spatial visualizations for the most significant CRE-cell type associations.

In [30]:
# Calculate calibration matrix for spatial plots
res_array = res['celltype_activity_array'].copy()
res_array = np.log1p(res_array)
res_array[np.isinf(res_array)] = np.nan

# Apply filtering
for cell_type in to_filter.index:
    res_array[:, res['celltype_activity'].index == cell_type, to_filter.loc[cell_type]] = np.nan

# Calculate calibration baseline
cre_mean = np.nanmean(res_array, axis=(0, 1))
cre_celltype_mean = np.nanmean(res_array, axis=0)

In [31]:
# Plot top CREs for each reproducible cell type
starrfish3.blacklist_cre = cre_blacklist
for celltype in reproducible_celltypes:
    cre_activity = res_df_fdc.loc[celltype]
    cre_q_values_right = res_q_right.loc[celltype]
    cre_q_values_left = res_q_left.loc[celltype]
    
    # Get significant CREs
    cre_q_values_right = cre_q_values_right[cre_q_values_right <= 0.05]
    cre_q_values_left = cre_q_values_left[cre_q_values_left <= 0.05]
    
    cre_right = cre_activity.loc[cre_q_values_right.index].sort_values(ascending=False).index
    cre_left = cre_activity.loc[cre_q_values_left.index].sort_values(ascending=True).index
    
    # Calculate nmax for color scaling
    if len(cre_right) > 0:
        nmax = starrfish3.get_cre_expression().loc[
            starrfish3.get_celltypes() == celltype, cre_right[0]].max()
        t7_nmax = starrfish3.get_t7_expression().loc[
            starrfish3.get_celltypes() == celltype, cre_right[0]].mean()
        nmax = np.log1p(nmax / t7_nmax) - cre_mean[res_df_fdc.columns == cre_right[0]][0]
        nmax = int(np.ceil(nmax))
        
        # Plot right tail (activated)
        cre = cre_right[0]
        print(f'Plotting {cre} in {celltype} (activated)')
        fig = starrfish3.plot_gene(
            cre, average_by_celltype=False,
            cell_types_to_visualize=[celltype],
            scale_size_by='counts',
            log=True, calibrate=cre_mean[res_df_fdc.columns == cre][0], nmax=nmax,
            transpose=-1, flipx=-1, sz_max=50,
            cell_types_to_use=[celltype]
        )
        fig.savefig(f'results/expr3/celltype_significant_cres/{celltype}_{cre}_right_tail.pdf')
        plt.close()
    
    # Plot left tail (repressed)
    if len(cre_left) > 0:
        cre = cre_left[0]
        print(f'Plotting {cre} in {celltype} (repressed)')
        fig = starrfish3.plot_gene(
            cre, average_by_celltype=False,
            cell_types_to_visualize=[celltype],
            scale_size_by='counts',
            log=True, calibrate=cre_mean[res_df_fdc.columns == cre][0], nmax=nmax,
            transpose=-1, flipx=-1, sz_max=50,
            cell_types_to_use=[celltype]
        )
        fig.savefig(f'results/expr3/celltype_significant_cres/{celltype}_{cre}_left_tail.pdf')
        plt.close()

Plotting CRE097 in Astro-NT NN (activated)
Plotting CRE262 in Astro-NT NN (repressed)
Plotting CRE322 in Astro-OLF NN (activated)
Plotting CRE254 in Astro-OLF NN (repressed)
Plotting CRE097 in Astro-TE NN (activated)
Plotting CRE163 in Astro-TE NN (repressed)
Plotting CRE343 in Bergmann NN (activated)
Plotting CRE256 in Bergmann NN (repressed)
Plotting CRE343 in CB Granule Glut (activated)
Plotting CRE255 in CB Granule Glut (repressed)
Plotting CRE383 in CBX MLI Megf11 Gaba (activated)
Plotting CRE173 in CBX MLI Megf11 Gaba (repressed)
Plotting CRE264 in CBX Purkinje Gaba (activated)
Plotting CRE190 in CBX Purkinje Gaba (repressed)
Plotting CRE167 in DG Glut (activated)
Plotting CRE021 in Endo NN (activated)
Plotting CRE228 in Endo NN (repressed)
Plotting CRE219 in IT AON-TT-DP Glut (activated)
Plotting CRE209 in IT AON-TT-DP Glut (repressed)
Plotting CRE229 in L2-3 IT CTX Glut (activated)
Plotting CRE298 in L2-3 IT CTX Glut (repressed)
Plotting CRE097 in L2-3 IT RSP Glut (activated)
P

## 13. ATAC-seq Validation

### 13.1 Precision/Recall Functions

In [32]:
def get_precision_df(res_q_df, starrfish, use='atac-peak'):
    """Calculate precision/recall of CRE activity vs ATAC-seq peaks.
    
    Returns DataFrame with TP, Total, precision, and recall for each cell type.
    """
    precision_df = pd.DataFrame(
        index=res_q_df.index, 
        columns=['TP', 'Total', 'ATAC', 'precision', 'recall']
    )
    
    for celltype in precision_df.index:
        atac_peaks = starrfish.get_positive_control_cres(cell_type=celltype, use=use)
        sig_cres = res_q_df.columns[res_q_df.loc[celltype] <= 0.05]
        nan_cres = res_q_df.columns[pd.isna(res_q_df.loc[celltype])]
        atac_peaks = atac_peaks[~atac_peaks.isin(nan_cres)] if atac_peaks is not None else None
        
        if atac_peaks is not None and len(sig_cres) > 0:
            precision_df.loc[celltype, 'TP'] = sig_cres.isin(atac_peaks).sum()
        
        precision_df.loc[celltype, 'Total'] = sum(~pd.isna(res_q_df.loc[celltype]))
        precision_df.loc[celltype, use] = len(atac_peaks) if atac_peaks is not None else 0
        
        if precision_df.loc[celltype, 'Total'] != 0:
            precision_df.loc[celltype, 'recall'] = precision_df.loc[celltype, 'TP'] / precision_df.loc[celltype, 'Total']
        if precision_df.loc[celltype, use] != 0:
            precision_df.loc[celltype, 'precision'] = precision_df.loc[celltype, 'TP'] / precision_df.loc[celltype, use]
    
    precision_df = precision_df.sort_values('precision', ascending=False)
    return precision_df


def plot_atac_precision(atac_precision_df, celltypes_to_use, use='atac-peak'):
    """Plot ATAC-seq precision across cell types."""
    cluster_annotation_term = pd.read_csv('Data/abc_atlas/cluster_annotation_term.csv', index_col=0)
    cluster_annotation_term['subclass'] = cluster_annotation_term['subclass'].str.replace('/', '-')
    atac_precision_df['cell_type_rank'] = cluster_annotation_term['subclass_number'].groupby(
        cluster_annotation_term['subclass']).first().loc[atac_precision_df.index].values
    atac_precision_df = atac_precision_df.sort_values(['cell_type_rank'], ascending=[True])
    
    atac_precision_df_filtered = atac_precision_df.loc[celltypes_to_use.intersection(atac_precision_df.index)]
    atac_precision_df_filtered = atac_precision_df_filtered.sort_values(['cell_type_rank'], ascending=[True])
    
    cell_types = atac_precision_df_filtered.index
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Left y-axis: precision
    sns.barplot(x=cell_types, y=atac_precision_df_filtered['precision'], ax=ax1, alpha=0.8)
    ax1.set_xlabel('Cell Types')
    ax1.set_ylabel('ATAC Precision', color='black')
    ax1.tick_params(axis='y', labelcolor='black')
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    
    # Overall precision line
    overall_precision = atac_precision_df_filtered['TP'].sum() / atac_precision_df_filtered[use].sum()
    ax1.axhline(y=overall_precision, color='black', linestyle='--', alpha=0.7, 
                label=f'Overall precision: {overall_precision:.3f}')
    ax1.legend(loc='upper left')
    
    # Right y-axis: counts
    ax2 = ax1.twinx()
    x_pos = np.arange(len(atac_precision_df_filtered))
    
    ax2.bar(x_pos - 0.2, atac_precision_df_filtered['TP'], 0.4, 
            label=f'Significant CREs in {use}', alpha=0.8, color='orange')
    ax2.bar(x_pos + 0.2, atac_precision_df_filtered[use], 0.4, 
            label=f'Total {use}', alpha=0.8, color='green')
    ax2.set_ylabel('Count', color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    return fig

### 13.2 Cell Type-Level ATAC Precision

In [33]:
# Calculate precision for multiple assays
overall_tp = {}
overall_total = {}

for use in ['atac-peak', 'h3k27ac-peak', 'h3k4me1-peak', 'chromatin-a']:
    precision_df = get_precision_df(res_q_right, starrfish3, use=use)
    fig = plot_atac_precision(precision_df, cell_types_to_use, use=use)
    fig.savefig(f'results/expr3/{use.replace("-", "_")}_precision_by_celltype.pdf')
    plt.close()
    
    overall_tp[use] = precision_df.loc[cell_types_to_use, 'TP'].sum()
    overall_total[use] = precision_df.loc[cell_types_to_use, use].sum()

print("Overall precision by assay:")
for use in overall_tp.keys():
    print(f"  {use}: {overall_tp[use] / overall_total[use]:.3f}")

Overall precision by assay:
  atac-peak: 0.084
  h3k27ac-peak: 0.085
  h3k4me1-peak: 0.071
  chromatin-a: 0.210


In [34]:
# Overall precision barplot
cre_precision_data = pd.DataFrame({
    'Assay': list(overall_tp.keys()),
    'TP': list(overall_tp.values()),
    'Total': list(overall_total.values()),
    'Precision': [tp / total if total > 0 else 0 
                  for tp, total in zip(overall_tp.values(), overall_total.values())]
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=cre_precision_data, x='Assay', y='Precision', ax=ax, alpha=0.8)
ax.set_xlabel('Assay')
ax.set_ylabel('Overall Precision')
ax.set_ylim(0, 0.3)
for i, row in cre_precision_data.iterrows():
    ax.text(i, row['Precision'] + 0.02, f"{row['TP']}/{row['Total']}", 
            ha='center', va='bottom')
fig.savefig('results/expr3/overall_precision_barplot.pdf')
plt.show()

## 14. Reproducible CRE-Cell Type Pairs

Filter for CRE-cell type associations that are significant in both sections.

In [35]:
# Create reproducible q-value dataframes
celltypes_overlap = res_q1_right.index.intersection(res_q2_right.index)
res_q_reproducible = (
    (res_q1_right.loc[celltypes_overlap] <= 0.05) & 
    (res_q2_right.loc[celltypes_overlap] <= 0.05)
)

# Combined dataset (only reproducible pairs)
res_q_right_reproducible = res_q_right.loc[celltypes_overlap].copy()
res_q_right_reproducible[~res_q_reproducible] = np.nan
res_df_fdc_reproducible = res_df_fdc.loc[celltypes_overlap].copy()
res_df_fdc_reproducible[res_q_right_reproducible.isna()] = np.nan

# Section 1 (only reproducible pairs)
res_q1_right_reproducible = res_q1_right.loc[celltypes_overlap].copy()
res_q1_right_reproducible[~res_q_reproducible] = np.nan
res_df1_fdc_reproducible = res_df1_fdc.loc[celltypes_overlap].copy()
res_df1_fdc_reproducible[res_q1_right_reproducible.isna()] = np.nan

# Section 2 (only reproducible pairs)
res_q2_right_reproducible = res_q2_right.loc[celltypes_overlap].copy()
res_q2_right_reproducible[~res_q_reproducible] = np.nan
res_df2_fdc_reproducible = res_df2_fdc.loc[celltypes_overlap].copy()
res_df2_fdc_reproducible[res_q2_right_reproducible.isna()] = np.nan

print(f"Total reproducible pairs: {res_q_reproducible.sum().sum()}")

Total reproducible pairs: 129


In [36]:
# Plot reproducible pairs
cell_types_to_use = res_q_right_reproducible.index[
    np.nanmin(res_q_right_reproducible, axis=1) < 0.05
]
cres_to_use = res_q_right_reproducible.columns[
    np.nanmin(res_q_right_reproducible.loc[cell_types_to_use], axis=0) < 0.05
].union(starrfish3.get_negative_control_cres())
cres_to_use = cres_to_use[~cres_to_use.isin(cre_blacklist)]

fig, final_order = celltype_pval_dotplot(
    res_q_right_reproducible, res_df_fdc_reproducible, cres_to_use, cell_types_to_use,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(12, 20)
)
fig.savefig('results/expr3/celltype_pval_dotplot_reproducible_CRE_CellType_pair.pdf')
plt.close()

# Section-specific plots
fig, _ = celltype_pval_dotplot(
    res_q1_right_reproducible, res_df1_fdc_reproducible, pd.Index(final_order), 
    cell_types_to_use, reorder_cres=False,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(12, 20)
)
fig.savefig('results/expr3/celltype_pval_dotplot_sec1_reproducible_CRE_CellType_pair.pdf')
plt.close()

fig, _ = celltype_pval_dotplot(
    res_q2_right_reproducible, res_df2_fdc_reproducible, pd.Index(final_order), 
    cell_types_to_use, reorder_cres=False,
    positive_control_info=cre_info, significant_cutoff=0.05, z_norm=False,
    figsize=(12, 20)
)
fig.savefig('results/expr3/celltype_pval_dotplot_sec2_reproducible_CRE_CellType_pair.pdf')
plt.close()

## 15. CRE-Level On-Target Analysis (Figure S9b, S9c)

Calculate how many CREs show activity in their expected target cell types.

In [37]:
def get_cre_precision_df(res_q_df, starrfish, celltypes_to_use=None, use='atac-peak'):
    """Calculate precision/recall for each CRE vs ATAC-seq data.
    
    Returns DataFrame with TP, Total, precision, and recall for each CRE.
    """
    precision_df = pd.DataFrame(
        index=res_q_df.columns, 
        columns=['TP', 'Total', 'precision', 'recall']
    )
    
    for cre in precision_df.index:
        atac_peaks = starrfish.get_positive_control_celltypes(cre=cre, use=use)
        sig_celltypes = res_q_df.index[res_q_df[cre] <= 0.05]
        nan_celltypes = res_q_df.index[pd.isna(res_q_df[cre])]
        
        if celltypes_to_use is not None:
            sig_celltypes = sig_celltypes[sig_celltypes.isin(celltypes_to_use)]
            nan_celltypes = nan_celltypes[nan_celltypes.isin(celltypes_to_use)]
        
        atac_peaks = atac_peaks[~atac_peaks.isin(nan_celltypes)] if atac_peaks is not None else None
        
        if atac_peaks is not None and len(sig_celltypes) > 0:
            precision_df.loc[cre, 'TP'] = sig_celltypes.isin(atac_peaks).sum()
        
        precision_df.loc[cre, 'Total'] = sum(~pd.isna(res_q_df[cre]))
        precision_df.loc[cre, use] = len(atac_peaks) if atac_peaks is not None else 0
        
        if precision_df.loc[cre, 'Total'] != 0:
            precision_df.loc[cre, 'recall'] = precision_df.loc[cre, 'TP'] / precision_df.loc[cre, 'Total']
        if precision_df.loc[cre, use] != 0:
            precision_df.loc[cre, 'precision'] = precision_df.loc[cre, 'TP'] / precision_df.loc[cre, use]
        
        precision_df.loc[cre, 'atac_peaks_celltypes'] = ', '.join(atac_peaks) if atac_peaks is not None else ''
    
    precision_df = precision_df.sort_values('precision', ascending=False)
    return precision_df

In [38]:
# Calculate CRE precision for reproducible CREs
reproducible_cres = overlap_cre_df.index[
    overlap_cre_df['reproducibility'].isin(['All Reproducible', 'Partially Reproducible'])
]

cre_precision_data_all = []

for use in ['atac-peak', 'h3k27ac-peak', 'h3k4me1-peak', 'chromatin-a']:
    for y in ['precision', 'percentage']:
        # Reproducible CREs
        cre_precision_df_repro = get_cre_precision_df(
            res_q_right[reproducible_cres].copy(), starrfish3, use=use)
        repro_percentage = sum(cre_precision_df_repro['TP'] > 0) / sum(~cre_precision_df_repro['TP'].isna())
        repro_precision = sum(cre_precision_df_repro['TP'] > 0) / sum(cre_precision_df_repro[use] > 0)
        cre_precision_df_repro.to_csv(f'results/expr3/cre_{y}_reproducible_{use.replace("-", "_")}_df.csv')
        
        # All CREs
        cre_precision_df_all = get_cre_precision_df(res_q_right.copy(), starrfish3, use=use)
        all_percentage = sum(cre_precision_df_all['TP'] > 0) / sum(~cre_precision_df_all['TP'].isna())
        all_precision = sum(cre_precision_df_all['TP'] > 0) / sum(cre_precision_df_all[use] > 0)
        cre_precision_df_all.to_csv(f'results/expr3/cre_{y}_all_{use.replace("-", "_")}_df.csv')
        
        repro_tp_count = sum(cre_precision_df_repro['TP'] > 0)
        repro_total_count = sum(~cre_precision_df_repro['TP'].isna())
        all_tp_count = sum(cre_precision_df_all['TP'] > 0)
        all_total_count = sum(~cre_precision_df_all['TP'].isna())
        
        print(f'{use} Reproducible CREs {y}: {repro_tp_count} out of {repro_total_count}')
        print(f'{use} All CREs {y}: {all_tp_count} out of {all_total_count}')
        
        cre_precision_data = pd.DataFrame({
            'CRE_type': ['Sec1-Sec2\nReproducible CREs', 'All CREs'],
            'percentage': [repro_percentage, all_percentage],
            'precision': [repro_precision, all_precision],
            'TP_count': [repro_tp_count, all_tp_count],
            'Total_count': [sum(cre_precision_df_repro[use] > 0), sum(cre_precision_df_all[use] > 0)],
            'use': [use, use],
            'y': [y, y]
        })
        cre_precision_data_all.append(cre_precision_data)
        
        # Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 4))
        
        # Percentage/Precision
        sns.barplot(data=cre_precision_data, x='CRE_type', y=y, ax=ax1, alpha=0.8)
        ax1.set_xlabel('CRE Categories')
        ax1.set_ylabel('Percentage' if y == 'percentage' else 'Precision')
        ax1.set_ylim(0, 1)
        
        if y == 'precision':
            ax1.set_title('Precision of CREs on-target')
            ax1.text(0, cre_precision_data.loc[0, y] + 0.02,
                    f"{repro_tp_count}/{sum(cre_precision_df_repro[use] > 0)}",
                    ha='center', va='bottom', fontsize=9)
            ax1.text(1, cre_precision_data.loc[1, y] + 0.02,
                    f"{all_tp_count}/{sum(cre_precision_df_all[use] > 0)}",
                    ha='center', va='bottom', fontsize=9)
        else:
            ax1.set_title('Percentage of CREs on-target')
            ax1.text(0, cre_precision_data.loc[0, y] + 0.02,
                    f"{repro_tp_count}/{repro_total_count}",
                    ha='center', va='bottom', fontsize=9)
            ax1.text(1, cre_precision_data.loc[1, y] + 0.02,
                    f"{all_tp_count}/{all_total_count}",
                    ha='center', va='bottom', fontsize=9)
        
        # Count
        sns.barplot(data=cre_precision_data, x='CRE_type', y='TP_count', ax=ax2, alpha=0.8, color='orange')
        ax2.set_xlabel('CRE Categories')
        ax2.set_ylabel('Number')
        ax2.set_title('Count of CREs on-target')
        for i, row in cre_precision_data.iterrows():
            ax2.text(i, row['TP_count'] + max(cre_precision_data['TP_count']) * 0.01,
                    f"{int(row['TP_count'])}",
                    ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        fig.savefig(f'results/expr3/cre_{y}_barplot_{use.replace("-", "_")}.pdf')
        plt.close()

atac-peak Reproducible CREs precision: 20 out of 122
atac-peak All CREs precision: 40 out of 356
atac-peak Reproducible CREs percentage: 20 out of 122
atac-peak All CREs percentage: 40 out of 356
h3k27ac-peak Reproducible CREs precision: 2 out of 122
h3k27ac-peak All CREs precision: 4 out of 356
h3k27ac-peak Reproducible CREs percentage: 2 out of 122
h3k27ac-peak All CREs percentage: 4 out of 356
h3k4me1-peak Reproducible CREs precision: 1 out of 122
h3k4me1-peak All CREs precision: 4 out of 356
h3k4me1-peak Reproducible CREs percentage: 1 out of 122
h3k4me1-peak All CREs percentage: 4 out of 356
chromatin-a Reproducible CREs precision: 8 out of 122
chromatin-a All CREs precision: 13 out of 356
chromatin-a Reproducible CREs percentage: 8 out of 122
chromatin-a All CREs percentage: 13 out of 356


In [39]:
# Summary plot for reproducible CREs
cre_precision_data = pd.concat(cre_precision_data_all, axis=0, ignore_index=True)
cre_precision_data = cre_precision_data[cre_precision_data['use'].isin(['atac-peak', 'chromatin-a'])]
cre_precision_data = cre_precision_data.drop_duplicates()
cre_precision_data = cre_precision_data[
    (cre_precision_data['CRE_type'] == 'Sec1-Sec2\nReproducible CREs') & 
    (cre_precision_data['y'] == 'precision')
]

fig, ax = plt.subplots(figsize=(3, 4))
sns.barplot(data=cre_precision_data, x='use', y='precision', ax=ax, alpha=0.8)
ax.set_xlabel('Assay')
ax.set_ylabel('Precision')
ax.set_ylim(0, max(cre_precision_data['precision']) + 0.1)
for idx, (i, row) in enumerate(cre_precision_data.iterrows()):
    ax.text(idx, row['precision'] + 0.01, f"{int(row['TP_count'])}/{int(row['Total_count'])}",
            ha='center', va='bottom', fontsize=9)
fig.savefig('results/expr3/cre_precision_reproducible_cres_barplot.pdf')
plt.show()

## 16. Multi-Assay Overlap (UpSet Plot)

Visualize overlap of on-target CREs across multiple chromatin assays.

In [40]:
# Collect on-target CREs for each assay
on_target_cres = {}
for use in ['atac-peak', 'h3k27ac-peak', 'h3k4me1-peak', 'chromatin-a']:
    cre_precision_df_all = get_cre_precision_df(
        res_q_right.copy(), starrfish3, reproducible_celltypes, use=use)
    on_target_cres[use] = cre_precision_df_all.index[cre_precision_df_all['TP'] > 0]
    overall_tp[use] = cre_precision_df_all['TP'].sum()
    overall_total[use] = len(cre_precision_df_all)

# Create UpSet plot
upset_data = from_contents(on_target_cres)
fig = plt.figure(figsize=(6, 4))
upset = UpSet(upset_data, subset_size='count', show_counts='%d', 
              sort_by='degree', sort_categories_by=None)
upset.plot(fig=fig)
fig.savefig('results/expr3/cre_ontarget_upsetplot.pdf')
plt.show()

print("\nOn-target CRE counts by assay:")
for use, cres in on_target_cres.items():
    print(f"  {use}: {len(cres)}")


On-target CRE counts by assay:
  atac-peak: 18
  h3k27ac-peak: 3
  h3k4me1-peak: 1
  chromatin-a: 9


## 17. Save activity matrix to csv

In [41]:
res_df_fdc.to_csv('results/expr3/cre_activity.csv')
res_q_right.to_csv('results/expr3/cre_q_values.csv')

## 18. Activity correlation with ATAC cpm

In [42]:
atac_cpm = starrfish3.atac_cpm.copy()
# overlap with res_df_fdc
atac_cpm = atac_cpm.loc[res_df_fdc.index.intersection(atac_cpm.index), res_df_fdc.columns.intersection(atac_cpm.columns)]
res_df_fdc_overlap_atac = res_df_fdc.copy().loc[atac_cpm.index, atac_cpm.columns]
# mark NaN values to atac_cpm where res_df_fdc is NaN
atac_cpm[res_df_fdc_overlap_atac.isna()] = np.nan
# flatten atac_cpm and res_df_fdc
atac_cpm_flat = atac_cpm.values.flatten()
# do log to atac_cpm
atac_cpm_flat = np.log1p(atac_cpm_flat)
res_df_fdc_flat = res_df_fdc_overlap_atac.values.flatten()
# do scatter plot
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(x=atac_cpm_flat, y=res_df_fdc_flat, alpha=0.1, ax=ax)
ax.set_xlabel('ATAC-seq CPM')
ax.set_ylabel('CRE Activity (FDC)')
ax.set_title('Correlation between ATAC-seq and CRE Activity')
# calculate correlation coefficient
valid_idx = ~pd.isna(atac_cpm_flat) & ~pd.isna(res_df_fdc_flat)
corr_coef = np.corrcoef(atac_cpm_flat[valid_idx], res_df_fdc_flat[valid_idx])[0, 1]
ax.text(0.05, 0.95, f'Correlation: {corr_coef:.3f}', transform=ax.transAxes, 
        ha='left', va='top', fontsize=12, bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
fig.savefig('results/expr3/atac_vs_cre_activity_scatterplot.pdf')

## Summary

This analysis has:
1. Quantified CRE activity across cell types using bootstrap resampling
2. Assessed reproducibility between biological replicates (tissue sections)
3. Validated findings against ATAC-seq and histone modification data
4. Identified high-confidence CRE-cell type associations
5. Generated spatial visualizations and comprehensive summary plots